Based on your notebook, here are all the tables and dataframes you're working with:

## Source Tables:
1. **`prod.ml.ip_vertical_associations`** - Production IP to vertical associations
2. **`dev.ml.ip_vertical_associations`** - Dev/new model IP to vertical associations
3. **S3 Parquet file**: `s3://mntn-data-archive-prod/vertical_categorizations/advertiser_verticals/` - Vertical reference data

## Created DataFrames:
1. **`prod_df`** - Filtered prod data
2. **`dev_df`** - Filtered dev data
3. **`verticals_df`** - Vertical reference data
4. **`prod_with_verticals`** - prod_df joined with vertical names
5. **`dev_with_verticals`** - dev_df joined with vertical names
6. **`prod_by_vertical`** - Aggregated prod counts by vertical
7. **`dev_by_vertical`** - Aggregated dev counts by vertical
8. **`comparison`** - Joined prod/dev counts with calculated changes
9. **`changed_ips`** - IPs that changed verticals
10. **`new_verticals`** - Verticals only in dev
11. **`lost_verticals`** - Verticals only in prod
12. **`vertical_churn`** - New analysis showing IP movement within verticals

## Temporary Views (for SQL):
1. **`prod_ips`** - SQL view of prod_with_verticals
2. **`dev_ips`** - SQL view of dev_with_verticals


In [0]:
# Dictionary of all dataframes
dataframes = {
    "prod_df": prod_df,
    "dev_df": dev_df,
    "verticals_df": verticals_df,
    "prod_with_verticals": prod_with_verticals,
    "dev_with_verticals": dev_with_verticals,
    "prod_by_vertical": prod_by_vertical,
    "dev_by_vertical": dev_by_vertical,
    #"comparison": comparison,
    #"changed_ips": changed_ips,
    #"new_verticals": new_verticals,
    #"lost_verticals": lost_verticals
}

# Loop through and display info
for name, df in dataframes.items():
    print(f"\n{'='*50}")
    print(f"DataFrame: {name}")
    print(f"{'='*50}")
    print(f"Columns: {df.columns}")
    print(f"Number of columns: {len(df.columns)}")
    print(f"Schema:")
    df.printSchema()


DataFrame: prod_df
Columns: ['ip', 'date', 'data_source_id', 'data_source_category_id', 'dt', 'vertical_id']
Number of columns: 6
Schema:
root
 |-- ip: string (nullable = true)
 |-- date: date (nullable = true)
 |-- data_source_id: integer (nullable = true)
 |-- data_source_category_id: double (nullable = true)
 |-- dt: string (nullable = true)
 |-- vertical_id: string (nullable = true)


DataFrame: dev_df
Columns: ['ip', 'date', 'data_source_id', 'data_source_category_id', 'dt', 'vertical_id']
Number of columns: 6
Schema:
root
 |-- ip: string (nullable = true)
 |-- date: date (nullable = true)
 |-- data_source_id: integer (nullable = true)
 |-- data_source_category_id: double (nullable = true)
 |-- dt: string (nullable = true)
 |-- vertical_id: string (nullable = true)


DataFrame: verticals_df
Columns: ['vertical_name', 'vertical_id']
Number of columns: 2
Schema:
root
 |-- vertical_name: string (nullable = true)
 |-- vertical_id: integer (nullable = true)


DataFrame: prod_with_vert

In [0]:
from pyspark.sql import functions as F

# Step 1: Extract Original Data (Prod)
print("\nExtract Prod Data")
print("=====================================")

# Query prod table
prod_df = spark.table('prod.ml.ip_vertical_associations') \
   .filter("dt = '2025-07-07'") \
   .withColumn('vertical_id', F.floor('data_source_category_id').cast('string')) \
   .filter(F.length(F.col('vertical_id')) == 6)

# Load verticals reference data
verticals_df = spark.read.parquet("s3://mntn-data-archive-prod/vertical_categorizations/advertiser_verticals/")

# Join to get vertical names
prod_with_verticals = prod_df.join(verticals_df, on='vertical_id', how='inner')

# Count total distinct IPs
prod_total_ips = prod_with_verticals.select('ip').distinct().count()
print(f"Total distinct IPs in prod: {prod_total_ips:,}")

# Get counts by vertical
prod_by_vertical = prod_with_verticals.groupBy('vertical_name') \
   .agg(F.countDistinct('ip').alias('ip_count')) \
   .orderBy('ip_count', ascending=False)

print("\nTop 10 verticals by IP count (Prod):")
prod_by_vertical.limit(10).show()



Extract Prod Data
Total distinct IPs in prod: 191,104,676

Top 10 verticals by IP count (Prod):
+--------------------+--------+
|       vertical_name|ip_count|
+--------------------+--------+
|Apparel & Accesso...|49260561|
|Internet Service ...|34791563|
|  Resale Marketplace|33455680|
|Consumer Electronics|27558610|
|            Footwear|26693198|
|            Skincare|22883581|
|       Food Products|21959963|
|Vitamins, Supplem...|21688154|
|Investments & Tra...|21198507|
|           Furniture|20308136|
+--------------------+--------+



In [0]:
# Step 2: Extract New Data (Dev)
print("\nExtract Dev Data")
print("==============================")

# Query dev table with same filters
dev_df = spark.table('dev.ml.ip_vertical_associations') \
   .filter("dt = '2025-07-07'") \
   .withColumn('vertical_id', F.floor('data_source_category_id').cast('string')) \
   .filter(F.length(F.col('vertical_id')) == 6)

# Join to get vertical names
dev_with_verticals = dev_df.join(verticals_df, on='vertical_id', how='inner')

# Count total distinct IPs
dev_total_ips = dev_with_verticals.select('ip').distinct().count()
print(f"Total distinct IPs in dev: {dev_total_ips:,}")

# Get counts by vertical
dev_by_vertical = dev_with_verticals.groupBy('vertical_name') \
   .agg(F.countDistinct('ip').alias('ip_count')) \
   .orderBy('ip_count', ascending=False)

print("\nTop 10 verticals by IP count (Dev):")
dev_by_vertical.limit(10).show()



Extract Dev Data
Total distinct IPs in dev: 209,192,040

Top 10 verticals by IP count (Dev):
+--------------------+--------+
|       vertical_name|ip_count|
+--------------------+--------+
|     Current Affairs|63147747|
|Apparel & Accesso...|43747648|
|      Games & Comics|34991764|
|Sewing, Arts, & C...|24641723|
|            Footwear|22459392|
|Vitamins, Supplem...|22016875|
| Music & Instruments|20509371|
|Apparel & Accesso...|20441912|
|Consumer Electronics|19655029|
|B2B - Sales & Mar...|19450606|
+--------------------+--------+



In [0]:
print("\nOverall Change")
print("==============================")

absolute_change = dev_total_ips - prod_total_ips
percent_change = (absolute_change / prod_total_ips) * 100

print(f"Prod distinct IPs: {prod_total_ips:>15,}")
print(f"Dev distinct IPs:  {dev_total_ips:>15,}")
print("-"*50)
print(f"Absolute change:   {absolute_change:>15,}")
print(f"Percentage change: {percent_change:>14.2f}%")

if absolute_change > 0:
    print(f"\n Dev has {absolute_change:,} more IPs than prod ({percent_change:.2f}% increase)")
else:
    print(f"\n Dev has {abs(absolute_change):,} fewer IPs than prod ({abs(percent_change):.2f}% decrease)")

# Migration Analysis
print("\nMigration Analysis")
print("==============================")

# Join prod and dev data to find IPs that changed verticals
changed_ips = prod_with_verticals.alias('prod').join(
    dev_with_verticals.alias('dev'),
    on='ip',
    how='inner'
).filter(F.col('prod.vertical_id') != F.col('dev.vertical_id'))

# Count total changed IPs
total_changed = changed_ips.select('ip').distinct().count()
print(f"IPs that changed verticals: {total_changed:>12,}")
print(f"Percentage of changes:      {(total_changed/prod_total_ips)*100:>11.2f}%")
print(f"IPs that stayed same:       {prod_total_ips - total_changed:>12,}")


Overall Change
Prod distinct IPs:     191,104,676
Dev distinct IPs:      209,192,040
--------------------------------------------------
Absolute change:        18,087,364
Percentage change:           9.46%

 Dev has 18,087,364 more IPs than prod (9.46% increase)

Migration Analysis
IPs that changed verticals:  147,061,810
Percentage of changes:            76.95%
IPs that stayed same:         44,042,866


In [0]:

# Step 4: Analyze Vertical-Level Changes
print("\nAnalyze Vertical-Level Changes")
print("======================================")

# Rename columns for clarity
prod_renamed = prod_by_vertical.withColumnRenamed('ip_count', 'prod_ips')
dev_renamed = dev_by_vertical.withColumnRenamed('ip_count', 'dev_ips')

# Join by vertical name
comparison = prod_renamed.join(dev_renamed, on='vertical_name', how='outer').fillna(0)

# Calculate changes
comparison = comparison.withColumn('absolute_change', F.col('dev_ips') - F.col('prod_ips')) \
   .withColumn('percent_change', 
       F.when(F.col('prod_ips') > 0, 
              (F.col('absolute_change') / F.col('prod_ips')) * 100
       ).otherwise(F.lit(None))
   )

# Top gainers
print("\nVerticals with MOST GAINS:")
comparison.filter(F.col('absolute_change') > 0) \
   .orderBy(F.col('absolute_change').desc()) \
   .select('vertical_name', 'prod_ips', 'dev_ips', 'absolute_change', 'percent_change') \
   .limit(10).show(truncate=False)

# Top losers
print("\nVerticals with MOST LOSSES:")
comparison.filter(F.col('absolute_change') < 0) \
   .orderBy(F.col('absolute_change').asc()) \
   .select('vertical_name', 'prod_ips', 'dev_ips', 'absolute_change', 'percent_change') \
   .limit(10).show(truncate=False)

# New verticals
print("\nNEW verticals (in dev but not prod):")
new_verticals = comparison.filter(F.col('prod_ips') == 0) \
   .select('vertical_name', 'dev_ips') \
   .orderBy('dev_ips', ascending=False)
new_verticals.show(truncate=False)

# Lost verticals
print("\nLOST verticals (in prod but not dev):")
lost_verticals = comparison.filter(F.col('dev_ips') == 0) \
   .select('vertical_name', 'prod_ips') \
   .orderBy('prod_ips', ascending=False)
lost_verticals.show(truncate=False)



Analyze Vertical-Level Changes

Verticals with MOST GAINS:
+-------------------------------------------+--------+--------+---------------+------------------+
|vertical_name                              |prod_ips|dev_ips |absolute_change|percent_change    |
+-------------------------------------------+--------+--------+---------------+------------------+
|Current Affairs                            |9403413 |63147747|53744334       |571.5407161208383 |
|Games & Comics                             |1113891 |34991764|33877873       |3041.3992931085713|
|Sewing, Arts, & Crafts                     |9331952 |24641723|15309771       |164.0575412303878 |
|Music & Instruments                        |8479435 |20509371|12029936       |141.8719053804882 |
|Learning & Eduction Technology             |3554864 |12927707|9372843        |263.66249172964143|
|Live Music & Comedy                        |4903517 |12076826|7173309        |146.2890615042224 |
|Real Estate & Builders                     |4830

In [0]:
# 1. Show schema (columns and data types)
vertical_churn.printSchema()

# 2. Get column names as a list
print(f"Columns: {vertical_churn.columns}")

root
 |-- vertical_name: string (nullable = true)
 |-- vertical_id: string (nullable = true)
 |-- prod_count: long (nullable = false)
 |-- dev_count: long (nullable = false)
 |-- retained_ips: long (nullable = false)
 |-- new_ips: long (nullable = false)
 |-- lost_ips: long (nullable = false)
 |-- pct_new_ips: decimal(27,2) (nullable = true)
 |-- pct_lost_ips: decimal(27,2) (nullable = true)

Columns: ['vertical_name', 'vertical_id', 'prod_count', 'dev_count', 'retained_ips', 'new_ips', 'lost_ips', 'pct_new_ips', 'pct_lost_ips']


In [0]:
# Step 4B: Vertical IP Churn Analysis (New Section)
print("\n" + "="*80)
print("VERTICAL IP CHURN ANALYSIS - Tracking Unique IPs Within Each Vertical")
print("="*80)

# Create temporary views for SQL analysis
prod_with_verticals.createOrReplaceTempView("prod_ips")
dev_with_verticals.createOrReplaceTempView("dev_ips")

# Analyze IP movements within each vertical
vertical_churn = spark.sql("""
WITH vertical_comparison AS (
    SELECT 
        COALESCE(p.vertical_name, d.vertical_name) as vertical_name,
        COALESCE(p.vertical_id, d.vertical_id) as vertical_id,
        COUNT(DISTINCT p.ip) as prod_count,
        COUNT(DISTINCT d.ip) as dev_count,
        COUNT(DISTINCT CASE WHEN p.ip IS NOT NULL AND d.ip IS NOT NULL THEN p.ip END) as retained_ips,
        COUNT(DISTINCT CASE WHEN p.ip IS NULL AND d.ip IS NOT NULL THEN d.ip END) as new_ips,
        COUNT(DISTINCT CASE WHEN p.ip IS NOT NULL AND d.ip IS NULL THEN p.ip END) as lost_ips
    FROM (
        SELECT DISTINCT vertical_id, vertical_name, ip FROM prod_ips
    ) p
    FULL OUTER JOIN (
        SELECT DISTINCT vertical_id, vertical_name, ip FROM dev_ips
    ) d
    ON p.vertical_id = d.vertical_id AND p.ip = d.ip
    GROUP BY COALESCE(p.vertical_name, d.vertical_name), COALESCE(p.vertical_id, d.vertical_id)
)
SELECT 
    vertical_name,
    vertical_id,  -- ADD THIS LINE
    prod_count,
    dev_count,
    retained_ips,
    new_ips,
    lost_ips,
    ROUND(CASE WHEN dev_count > 0 THEN (new_ips * 100.0 / dev_count) ELSE 0 END, 2) as pct_new_ips,
    ROUND(CASE WHEN prod_count > 0 THEN (lost_ips * 100.0 / prod_count) ELSE 0 END, 2) as pct_lost_ips
FROM vertical_comparison
ORDER BY new_ips DESC
""")

print("\nVERTICALS BY NUMBER OF NEW IPs (Top 20):")
print("Shows IPs that are NEW to each vertical (not just moved between verticals)")
vertical_churn.limit(20).show(truncate=False)

# Cache for follow-up analysis
vertical_churn.cache()

# Step 4C: Where Did New IPs Come From? (Follow-up Analysis)
print("\n" + "="*80)
print("FOLLOW-UP: WHERE DID THE NEW IPs COME FROM?")
print("="*80)

# Analyze top 5 verticals with most new IPs
top_verticals_with_new_ips = vertical_churn.orderBy(F.col('new_ips').desc()).limit(5).collect()

for row in top_verticals_with_new_ips:
    vertical_name = row['vertical_name']
    vertical_id = row['vertical_id']
    new_ip_count = row['new_ips']
    
    if new_ip_count > 0:
        print(f"\n--- {vertical_name} (gained {new_ip_count:,} new IPs) ---")
        
        # Find where these new IPs came from
        origin_analysis = spark.sql(f"""
            WITH new_ips AS (
                SELECT DISTINCT d.ip
                FROM dev_ips d
                LEFT JOIN prod_ips p ON d.ip = p.ip AND d.vertical_id = p.vertical_id
                WHERE d.vertical_id = '{vertical_id}' AND p.ip IS NULL
            )
            SELECT 
                p.vertical_name as from_vertical,
                COUNT(DISTINCT p.ip) as ip_count,
                ROUND(COUNT(DISTINCT p.ip) * 100.0 / {new_ip_count}, 2) as pct_of_new_ips
            FROM prod_ips p
            INNER JOIN new_ips n ON p.ip = n.ip
            GROUP BY p.vertical_name
            ORDER BY ip_count DESC
            LIMIT 10
        """)
        
        print("Top 10 sources for these new IPs:")
        origin_analysis.show(truncate=False)
        
        # Check completely new IPs
        brand_new_count = spark.sql(f"""
            WITH new_ips AS (
                SELECT DISTINCT d.ip
                FROM dev_ips d
                LEFT JOIN prod_ips p ON d.ip = p.ip AND d.vertical_id = p.vertical_id
                WHERE d.vertical_id = '{vertical_id}' AND p.ip IS NULL
            )
            SELECT COUNT(DISTINCT n.ip) as count
            FROM new_ips n
            LEFT JOIN (SELECT DISTINCT ip FROM prod_ips) all_prod ON n.ip = all_prod.ip
            WHERE all_prod.ip IS NULL
        """).collect()[0]['count']
        
        pct_brand_new = (brand_new_count / new_ip_count * 100) if new_ip_count > 0 else 0
        print(f"\nBrand new IPs (not in prod at all): {brand_new_count:,} ({pct_brand_new:.2f}% of new IPs)")

# Update Summary Statistics to include churn information
print("\n" + "="*80)
print("VERTICAL CHURN SUMMARY")
print("="*80)

# Get summary statistics from churn analysis
total_new_ips = vertical_churn.agg(F.sum('new_ips')).collect()[0][0]
total_lost_ips = vertical_churn.agg(F.sum('lost_ips')).collect()[0][0]
avg_churn_rate = vertical_churn.agg(F.avg('pct_lost_ips')).collect()[0][0]

print(f"\nOVERALL IP MOVEMENT ACROSS VERTICALS:")
print(f"- Total new IP-vertical associations: {total_new_ips:,}")
print(f"- Total lost IP-vertical associations: {total_lost_ips:,}")
print(f"- Average vertical churn rate: {avg_churn_rate:.2f}%")

# Verticals with highest churn
print(f"\nVERTICALS WITH HIGHEST CHURN RATES (Top 10):")
vertical_churn.filter(F.col('prod_count') > 1000) \
    .orderBy(F.col('pct_lost_ips').desc()) \
    .select('vertical_name', 'prod_count', 'dev_count', 'retained_ips', 'pct_lost_ips') \
    .limit(10).show(truncate=False)


VERTICAL IP CHURN ANALYSIS - Tracking Unique IPs Within Each Vertical

VERTICALS BY NUMBER OF NEW IPs (Top 20):
Shows IPs that are NEW to each vertical (not just moved between verticals)


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
# Step 5: Summary Statistics
print("\nSummary Statistics")
print("==========================")

# Overall count change and percentage impact
print(f"\nOVERALL COUNT CHANGE AND PERCENTAGE IMPACT:")
print(f"- Prod total IPs: {prod_total_ips:,}")
print(f"- Dev total IPs: {dev_total_ips:,}")
print(f"- Change: {absolute_change:,} IPs ({percent_change:+.2f}%)")

# IPs that changed verticals
print(f"\nIPs THAT CHANGED VERTICALS:")
print(f"- IPs with vertical changes: {total_changed:,}")
print(f"- Percentage of IPs changed: {(total_changed/prod_total_ips)*100:.2f}%")

# Highlight top gaining and losing verticals
top_gainer = comparison.orderBy(F.col('absolute_change').desc()).first()
top_loser = comparison.orderBy(F.col('absolute_change').asc()).first()

print(f"\nTOP GAINING AND LOSING VERTICALS:")
if top_gainer and top_gainer['absolute_change'] > 0:
    print(f"- Biggest gainer: {top_gainer['vertical_name']} (+{top_gainer['absolute_change']:,} IPs)")
    print(f"  Prod: {top_gainer['prod_ips']:,} → Dev: {top_gainer['dev_ips']:,}")
if top_loser and top_loser['absolute_change'] < 0:
    print(f"- Biggest loser: {top_loser['vertical_name']} ({top_loser['absolute_change']:,} IPs)")
    print(f"  Prod: {top_loser['prod_ips']:,} → Dev: {top_loser['dev_ips']:,}")

# Verticals added or removed
print(f"\nVERTICALS ADDED OR REMOVED:")
print(f"- Verticals added: {new_verticals.count()}")
print(f"- Verticals removed: {lost_verticals.count()}")


#